# Fase 4: Evaluación Cuantitativa (RAGAS + Métricas Deterministas) y Auditoría Empírica — AiHoreca

Este notebook ejecuta y audita empíricamente el sistema **AiHoreca (RAG Agéntico Híbrido con Jerarquía Normativa y Memoria Multi-Sesión SQLite)** importando directamente los módulos de producción de `src/`:
- `src.evaluation.audit_corpus_and_index_stats`: Auditoría estructural de los **648 fragmentos jerárquicos** indexados en Qdrant local.
- `src.evaluation.get_golden_dataset` y `src.evaluation.evaluate_golden_dataset`: Evaluación sobre los **12 casos del Golden Dataset** distribuidos en 3 bloques (`guardrail_routing`, `provincial_rag`, `multi_turn_memory`).


In [1]:
import json
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import (
    audit_corpus_and_index_stats,
    evaluate_golden_dataset,
    get_golden_dataset,
)

corpus_stats = audit_corpus_and_index_stats()
golden_dataset = get_golden_dataset()
eval_report = evaluate_golden_dataset(force_recompute=False)

print(f"Total fragmentos auditados en Qdrant local: {corpus_stats['total_chunks']}")
print(f"Total casos en Golden Dataset: {len(golden_dataset)}")
print(f"Casos evaluados (Generativos: {eval_report.generative_cases} | Guardrails/HITL: {eval_report.guardrail_and_hitl_cases})")


d:\Proyectos\AiHoreca\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total fragmentos auditados en Qdrant local: 648
Total casos en Golden Dataset: 12
Casos evaluados (Generativos: 7 | Guardrails/HITL: 5)


## 1. Auditoría Estructural del Corpus e Índice Híbrido Qdrant (648 Chunks)
Desglose cuantitativo por documento normativo, capa jerárquica (`estatuto_trabajadores`, `aleh_estatal`, `convenio_provincial`), territorio normalizado y tipo de contenido (`articulo`, `tabla_salarial`, `anexo_o_disposicion`).


In [2]:
print("=== MATRIZ DOCUMENTO x TIPO DE CONTENIDO ===")
header = f"{'Documento Markdown':<28} | {'Artículos':>10} | {'Tablas Sal.':>11} | {'Anexos/Disp.':>12} | {'Total':>7}"
print(header)
print("-" * len(header))
for doc_name, counts in corpus_stats["doc_type_matrix"].items():
    print(
        f"{doc_name:<28} | {counts['articulo']:>10} | {counts['tabla_salarial']:>11} | "
        f"{counts['anexo_o_disposicion']:>12} | {counts['total']:>7}"
    )

print("\n=== DISTRIBUCIÓN POR CAPA NORMATIVA ===")
for layer, cnt in corpus_stats["by_normative_layer"].items():
    pct = (cnt / corpus_stats["total_chunks"]) * 100
    print(f"  - {layer:<24}: {cnt:>4} chunks ({pct:5.1f}%)")

print("\n=== ESTADÍSTICAS DE LONGITUD DE CHUNKS (CARACTERES) ===")
print(json.dumps(corpus_stats["char_length_stats"], indent=2))


=== MATRIZ DOCUMENTO x TIPO DE CONTENIDO ===
Documento Markdown           |  Artículos | Tablas Sal. | Anexos/Disp. |   Total
--------------------------------------------------------------------------------
aleh_vi_estatal.md           |         30 |           4 |            0 |      34
convenio_catalunya_bcn.md    |         83 |          25 |           40 |     148
convenio_leon_texto.md       |         60 |          19 |            0 |      79
convenio_madrid.md           |         74 |          47 |           15 |     136
estatuto_trabajadores_consolidado.md |        204 |           0 |           47 |     251

=== DISTRIBUCIÓN POR CAPA NORMATIVA ===
  - aleh_estatal            :   34 chunks (  5.2%)
  - convenio_provincial     :  363 chunks ( 56.0%)
  - estatuto_trabajadores   :  251 chunks ( 38.7%)

=== ESTADÍSTICAS DE LONGITUD DE CHUNKS (CARACTERES) ===
{
  "min": 175,
  "p25": 806,
  "median": 1632,
  "mean": 1651.1,
  "p75": 2716,
  "max": 2897
}


## 2. Resumen Ejecutivo de Métricas Globales (Deterministas + Juez RAGAS)
Evaluación integral en cuatro planos: **Enrutamiento y Guardrails**, **Recuperación Híbrida Multicapa**, **Fidelidad Numérica (Cero Alucinación)** y **Juez Semántico RAGAS (`RAGASJudgeSchema`)**.


In [3]:
summary_metrics = [
    ("Exactitud de Intención (Intent Accuracy)", f"{eval_report.intent_accuracy * 100:.2f}%"),
    ("Exactitud de Ruta Agéntica (Route Accuracy)", f"{eval_report.route_accuracy * 100:.2f}%"),
    ("Exactitud Territorial / Anti-Poisoning", f"{eval_report.territorio_accuracy * 100:.2f}%"),
    ("Cobertura de las 3 Capas Normativas (Layer Coverage)", f"{eval_report.mean_layer_coverage * 100:.2f}%"),
    ("Recall de Citas/Artículos/Tablas @8 (Citation Recall@8)", f"{eval_report.mean_citation_recall_at_k * 100:.2f}%"),
    ("Impacto de Cifras Ground Truth (GT Number Hit Rate)", f"{eval_report.mean_ground_truth_number_hit * 100:.2f}%"),
    ("Tasa de Cero Alucinación Numérica (Guardrail Nodo 5)", f"{eval_report.zero_hallucination_rate * 100:.2f}%"),
    ("RAGAS Faithfulness (Fidelidad al Contexto)", f"{eval_report.mean_faithfulness * 100:.2f}%"),
    ("RAGAS Answer Relevancy (Relevancia de Respuesta)", f"{eval_report.mean_answer_relevancy * 100:.2f}%"),
    ("RAGAS Context Recall Semántico", f"{eval_report.mean_context_recall_semantic * 100:.2f}%"),
    ("RAGAS Hierarchy Application Score (Art. 3.3 ET)", f"{eval_report.mean_hierarchy_application_score * 100:.2f}%"),
]

print("=== MÉTRICAS GLOBALES DEL BENCHMARK AIHORECA (12 CASOS) ===")
for name, val in summary_metrics:
    print(f"  * {name:<56}: {val:>8}")

print("\n=== PERFILADO DE LATENCIAS POR NODO (MEDIA MS) ===")
print(f"  * Nodo 1 (analyze_and_route)        : {eval_report.mean_routing_latency_ms:>8.1f} ms")
print(f"  * Nodo 3 (parallel_retrieval)       : {eval_report.mean_retrieval_latency_ms:>8.1f} ms")
print(f"  * Nodo 4+5 (legal_synthesis_audit)  : {eval_report.mean_synthesis_latency_ms:>8.1f} ms")
print(f"  * Latencia Total Media por Turno    : {eval_report.mean_total_latency_ms:>8.1f} ms")


=== MÉTRICAS GLOBALES DEL BENCHMARK AIHORECA (12 CASOS) ===
  * Exactitud de Intención (Intent Accuracy)                :  100.00%
  * Exactitud de Ruta Agéntica (Route Accuracy)             :  100.00%
  * Exactitud Territorial / Anti-Poisoning                  :  100.00%
  * Cobertura de las 3 Capas Normativas (Layer Coverage)    :  100.00%
  * Recall de Citas/Artículos/Tablas @8 (Citation Recall@8) :  100.00%
  * Impacto de Cifras Ground Truth (GT Number Hit Rate)     :   92.86%
  * Tasa de Cero Alucinación Numérica (Guardrail Nodo 5)    :  100.00%
  * RAGAS Faithfulness (Fidelidad al Contexto)              :   98.57%
  * RAGAS Answer Relevancy (Relevancia de Respuesta)        :  100.00%
  * RAGAS Context Recall Semántico                          :   97.14%
  * RAGAS Hierarchy Application Score (Art. 3.3 ET)         :  100.00%

=== PERFILADO DE LATENCIAS POR NODO (MEDIA MS) ===
  * Nodo 1 (analyze_and_route)        :   3899.3 ms
  * Nodo 3 (parallel_retrieval)       :   1472.2 ms
  *

## 3. Desglose Caso por Caso del Golden Dataset (`GD_01` a `GD_12`)
Auditoría individual de cada turno evaluado, incluyendo verificación de ruta, territorio activo, cobertura de capas, fidelidad numérica del Nodo 5 y puntuaciones del Juez RAGAS.


In [4]:
row_hdr = (
    f"{'Case ID':<38} | {'Ruta':<18} | {'Territorio':<10} | "
    f"{'Capas':>5} | {'Cit@8':>5} | {'0-Halluc':>8} | {'Faith':>5} | {'Relev':>5} | {'CtxRec':>6} | {'Hier':>5}"
)
print(row_hdr)
print("-" * len(row_hdr))
for c in eval_report.cases:
    terr_str = str(c.predicted_territorio or "None")
    print(
        f"{c.case_id:<38} | {c.predicted_route:<18} | {terr_str:<10} | "
        f"{c.layer_coverage_score:>5.2f} | {c.citation_recall_at_k:>5.2f} | "
        f"{str(c.zero_numerical_hallucination):>8} | {c.faithfulness:>5.2f} | "
        f"{c.answer_relevancy:>5.2f} | {c.context_recall_semantic:>6.2f} | {c.hierarchy_application_score:>5.2f}"
    )


Case ID                                | Ruta               | Territorio | Capas | Cit@8 | 0-Halluc | Faith | Relev | CtxRec |  Hier
------------------------------------------------------------------------------------------------------------------------------------
GD_01_OOS_TRAFICO                      | block_out_of_scope | None       |  1.00 |  1.00 |     True |  1.00 |  1.00 |   1.00 |  1.00
GD_02_OOS_OTRO_SECTOR                  | block_out_of_scope | None       |  1.00 |  1.00 |     True |  1.00 |  1.00 |   1.00 |  1.00
GD_03_UNSUPPORTED_SEVILLA              | block_out_of_scope | sevilla    |  1.00 |  1.00 |     True |  1.00 |  1.00 |   1.00 |  1.00
GD_04_HITL_FRIEGAPLATOS_SIN_PROVINCIA  | ask_clarification  | None       |  1.00 |  1.00 |     True |  1.00 |  1.00 |   1.00 |  1.00
GD_05_MADRID_HITL_RESOLUCION           | parallel_retrieve  | madrid     |  1.00 |  1.00 |     True |  1.00 |  1.00 |   1.00 |  1.00
GD_06_MADRID_SEGUIMIENTO_NOCTURNIDAD   | parallel_retrieve  | madrid 

## 4. Inspección de Cifras Verificadas y Dictámenes del Juez RAGAS en Casos Generativos
Detalle de los números verificados literalmente por el Guardrail de Salida (Nodo 5), citas recuperadas y justificación técnica emitida por el Juez Imparcial RAGAS.


In [5]:
for c in eval_report.cases:
    if c.predicted_route == "parallel_retrieve":
        print("=" * 95)
        print(f"CASO: {c.case_id} | Sesión: {c.session_id} | Territorio: {c.predicted_territorio}")
        print(f"Consulta Resuelta : {c.effective_query}")
        print(f"Categoría VI ALEH : {c.categoria_aleh_detectada}")
        print(f"Cifras Verificadas: {c.verified_numbers}")
        print(f"Cifras No Verif.  : {c.unverified_numbers}")
        print(f"Dictamen RAGAS    : {c.judge_critique}")


CASO: GD_05_MADRID_HITL_RESOLUCION | Sesión: eval-ses-madrid | Territorio: madrid
Consulta Resuelta : Soy friegaplatos a jornada completa en Madrid, ¿cuánto tengo que cobrar de sueldo base y cuántos días de vacaciones me tocan según el convenio de hostelería de Madrid?
Categoría VI ALEH : Auxiliar de limpieza / Marmitón / Freganchín (Área Funcional 2ª Cocina o 3ª Restauración, Grupo Profesional III)
Cifras Verificadas: ['1.101,69', '1.032,48', '1.130,43', '183,87', '55,60', '1.049,23', '983,31', '1.076,60', '175,11', '52,95']
Cifras No Verif.  : []
Dictamen RAGAS    : La respuesta es excelente, precisa y exhaustiva. Utiliza correctamente las tablas salariales vigentes (2024) y las históricas (2023) presentes en el contexto, respetando la jerarquía normativa y respondiendo con exactitud a los puntos solicitados sobre salario y vacaciones.
CASO: GD_06_MADRID_SEGUIMIENTO_NOCTURNIDAD | Sesión: eval-ses-madrid | Territorio: madrid
Consulta Resuelta : Soy friegaplatos a jornada completa en M